**【课程】**：[100天风控专家](https://bzavt.xetlk.com/s/2Y1t7x)
**【店铺】**：[东哥讲风控](https://app7hmmvkwr2019.h5.xiaoeknow.com)
**【作者】**：东哥起飞

In [33]:
import os
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

# 生成pmml
from sklearn2pmml import sklearn2pmml, PMMLPipeline
from sklearn_pandas import DataFrameMapper

# 加载pmml
from pypmml import Model

## 0.安装

### sklearn2pmml

`sklearn2pmml` 是 `sklearn2pmml` 库里的一个核心转换函数，它的作用就是把训练好的 scikit-learn 管道（Pipeline）对象，转换成一个标准的 PMML 文件，方便在其他支持 PMML 的系统里部署模型。

### pypmml

需要用到`pypmml`，而`pypmml`需要在本地安装并配置好 Java 环境（JDK）。

因为`pypmml`库本质上是一个 Python 封装器，它在后台会调用 Java 的 PMML 解析和执行引擎，所以必须依赖 Java 运行时环境才能工作。

因此安装pypmml的步骤是：

- 本地配置JAVA环境，安装JDK包
JDK包下载地址：https://www.oracle.com/java/technologies/downloads/

- pip install pypmml

In [95]:
# 版本
import pypmml as pp
import sklearn2pmml as sk2
import sklearn_pandas as skp
import sklearn as sk
print('pandas: %s' %pd.__version__)
print('numpy: %s' %np.__version__)
print('sklearn: %s' %sk.__version__)
print('lightgbm: %s' %lgb.__version__)
print('pypmml: %s' %pp.__version__)
print('sklearn2pmml: %s' %sk2.__version__)
print('sklearn_pandas: %s' %skp.__version__)

pandas: 2.0.0
numpy: 1.23.5
sklearn: 1.3.2
lightgbm: 3.3.1
pypmml: 0.9.17
sklearn2pmml: 0.104.0
sklearn_pandas: 2.2.0


## 1.导入数据

In [96]:
# 1. 导入数据
df = pd.read_excel('data_sample.xlsx')
print(f"数据形状: {df.shape}")

# 2. 等级编码
grade_mapping = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
df['repayment_ability_rank'] = df['repayment_ability_rank'].map(grade_mapping)
df['dev_stability_grade'] = df['dev_stability_grade'].map(grade_mapping)

# 3. 划分特征和目标
feature_names = [col for col in df.columns if col != 'y']
X = df[feature_names]
y = df['y']

print(f"特征矩阵: {X.shape}")
print(f"目标变量: {y.shape}")
print(f"特征列表: {feature_names}")

数据形状: (50000, 7)
特征矩阵: (50000, 6)
目标变量: (50000,)
特征列表: ['ovd_order_days_6m_grade', 'ovd_order_cnt_6m_grade', 'repayment_ability_rank', 'dev_stability_grade', 'have_fang_prob_grade', 'positive_biz_cnt_1y_grade']


## 2.训练LGB模型

In [97]:
# ==================== 数据划分 ====================
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # 推荐：分层采样，保持类别分布
)

# ==================== 模型参数配置 ====================
lgb_params = {
    'learning_rate': 0.05,          # 学习率
    'n_estimators': 500,           # 迭代次数
    'num_leaves': 8,                # 叶子节点数
    'min_child_samples': 50,        # 叶子节点最小样本数
    'subsample': 0.7,               # 样本采样比例
    'colsample_bytree': 0.8,        # 特征采样比例
    'random_state': 42,             # 随机种子（保证可复现）
    'n_jobs': -1,                   # 使用所有CPU核心
}

# ==================== 模型训练 ====================
lgb_model = lgb.LGBMClassifier(**lgb_params)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    eval_metric='auc',
    early_stopping_rounds=50,
    verbose=50,                     # 每20轮输出一次日志
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]  # 推荐：更明确的回调
)

# ==================== 特征列表 ====================
feature_names = list(X.columns)

# ==================== 模型信息输出 ====================
print(f"训练集大小: {len(X_train)} 样本")
print(f"验证集大小: {len(X_valid)} 样本")
print(f"特征数量: {len(feature_names)}")
print(f"模型参数: {lgb_params}")

# 最佳迭代次数
print(f"最佳迭代轮数: {lgb_model.best_iteration_}")
print(f"最佳验证集AUC: {lgb_model.best_score_['valid_1']['auc']:.4f}")

# 得到预测结果
y_pred_prob = lgb_model.predict_proba(X)[:, 1]

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.833774	training's binary_logloss: 0.243743	valid_1's auc: 0.833583	valid_1's binary_logloss: 0.245931
[50]	training's auc: 0.833774	training's binary_logloss: 0.243743	valid_1's auc: 0.833583	valid_1's binary_logloss: 0.245931
[100]	training's auc: 0.835371	training's binary_logloss: 0.241086	valid_1's auc: 0.834225	valid_1's binary_logloss: 0.243982
[100]	training's auc: 0.835371	training's binary_logloss: 0.241086	valid_1's auc: 0.834225	valid_1's binary_logloss: 0.243982
[150]	training's auc: 0.836125	training's binary_logloss: 0.240241	valid_1's auc: 0.834366	valid_1's binary_logloss: 0.243835
[150]	training's auc: 0.836125	training's binary_logloss: 0.240241	valid_1's auc: 0.834366	valid_1's binary_logloss: 0.243835
[200]	training's auc: 0.836766	training's binary_logloss: 0.23959	valid_1's auc: 0.834742	valid_1's binary_logloss: 0.24375
[200]	training's auc: 0.836766	training's binary_logloss: 0.2

## 3.生成PMML

In [98]:
def save_model_to_pmml(model, feature_names, output_path):
    """
    将训练好的模型保存为PMML格式文件
    
    Parameters
    ----------
    model : 训练好的模型对象
    feature_names : list
        模型使用的特征名称列表
    output_path : str
        PMML文件保存路径（如：'model.pmml'）
    """
    # 创建特征映射器
    feature_mapper = DataFrameMapper([([feat], None) for feat in feature_names])
    
    # 构建PMML管道
    pmml_pipeline = PMMLPipeline([
        ('feature_mapper', feature_mapper), 
        ('classifier', model)
    ])
    
    # 导出为PMML文件
    sklearn2pmml(pmml_pipeline, output_path, with_repr=True)
    
    print(f'模型已成功导出为PMML文件: {output_path}')

In [99]:
# 生成PMML
output_path = 'lgb_model.pmml'
save_model_to_pmml(lgb_model, feature_names, output_path)
# 保存变量信息
pd.DataFrame({'col':feature_names}).to_csv('feature_info.txt',index=0)

模型已成功导出为PMML文件: lgb_model.pmml


## 4.加载PMML

In [100]:
import math

def get_pmml_result(dataframe, model_name, feature_name):
    """
    加载PMML模型计算输出概率结果
    
    Parameters
    ----------
    dataframe : DataFrame
        数据集
    model_name : str
        PMML模型名字
    feature_name : str
        入模变量文件名字
    
    Returns
    -------
    list
        模型输出概率结果
    """
    # 加载PMML
    pmml_model = Model.load(model_name)
    
    # 加载变量列表
    feature_names = list(pd.read_csv(feature_name)['col'])
    
    # 准备特征数据
    features_data = dataframe.loc[:, feature_names]
    
    # 模型预测
    predictions = pmml_model.predict(features_data)
    
    # 提取概率
    probabilities = list(predictions['probability(1)'])
    
    return probabilities

In [101]:
pmml_predict = get_pmml_result(X, 'lgb_model.pmml', 'feature_info.txt')

In [102]:
# 模型训练预测结果，与加载PMML后的预测结果，对比
pd.DataFrame({'lgb_predict':y_pred_prob,'pmml_predict':pmml_predict})

,lgb_predict,pmml_predict
0,0.029095,0.029095
1,0.025861,0.025861
2,0.043156,0.043156
3,0.034933,0.034933
4,0.025429,0.025429
...,...,...
49995,0.094201,0.094201
49996,0.104110,0.104110
49997,0.787375,0.787375
49998,0.104110,0.104110


## At Last.好课推荐

更多风控好课👉[东哥讲风控](https://app7hmmvkwr2019.h5.xiaoeknow.com)

### 1）贷前策略实战项目（提高班）

贷前策略实战项目，4大真实贷前场景实战项目

内容详情介绍👉[《贷前策略实战项目（提高班）》](https://mp.weixin.qq.com/s/33nsHN3v_Zf9sPkr-yp_iA)，下单链接👉[点这里](https://bzavt.xetlk.com/s/4io33W)

### 2）pandas进阶宝典

Pandas数据分析的各种骚操作，东哥的原创笔记， 500页飞书图文笔记，近30万字，已经完全体。配套完整代码支持下载，永久访问权限。

内容详情介绍👉[《pandas进阶宝典》](https://mp.weixin.qq.com/s/9WbDFamcK2WywagdfN_f9Q)，包括5大核心图文，下单链接👉[点这里](https://app7hmmvkwr2019.h5.xiaoeknow.com/p/course/ecourse/course_2YD5u0x8FzrAIyM8soEuxnTkP9r)
- 《pandas快速入门》
- 《pandas进阶宝典》
- 《pandas实战项目》
- 《pandas进阶题库》
- 《Numpy速查手册》
- 《正则表达式手册》

以上如有不清楚也可加我微信：`Petery_1966` 咨询